In [1]:
import pandas as pd

from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

c:\Users\PC\anaconda3\envs\skripsi\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
df = pd.read_csv("data/processed/clean_data.csv")

print(df.shape)
df.head()

(12000, 3)


,clean_text,score,bank
0,dikit verivikasi ribet mohon di perbaiki,1,BCAMobile_reviews
1,sangat terbantu sexali dg adanya m bangking in,1,BCAMobile_reviews
2,kenapa sy tidak bisa mendownlod aplikasi bca,1,BCAMobile_reviews
3,sangat kecewa saya melakukan transfer keterang...,1,BCAMobile_reviews
4,update an baru ada bugs,1,BCAMobile_reviews


In [3]:
texts = df["clean_text"].astype(str).tolist()

# IndoBERT

In [4]:
model = SentenceTransformer(
  "indobenchmark/indobert-base-p1",
  device="cuda" # using rtx 3060
)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 14214.83it/s]


In [5]:
embeddings = model.encode(texts,batch_size=32,show_progress_bar=True)

Batches:   0%|          | 0/375 [00:00<?, ?it/s]

Batches: 100%|██████████| 375/375 [00:12<00:00, 29.53it/s]


In [6]:
# sanity check
sim = cosine_similarity([embeddings[0]], embeddings[:10])

print(sim)

# expected result
# nilai mendekati 1 = mirip
# lebih kecil = beda

[[0.9999999  0.47826347 0.5376282  0.5520282  0.64250004 0.501776
  0.5075089  0.44884416 0.5895494  0.5600138 ]]


| Nilai similarity | Arti                |
| ---------------- | ------------------- |
| ~0.9             | hampir identik      |
| 0.6 – 0.8        | mirip               |
| 0.4 – 0.6        | agak mirip (normal) |
| <0.3             | beda                |


In [7]:
print(texts[0])
print(texts[1])

dikit verivikasi ribet mohon di perbaiki
sangat terbantu sexali dg adanya m bangking in


# BERTopic

In [8]:
from bertopic import BERTopic

In [9]:
topic_model = BERTopic()

topics, probs = topic_model.fit_transform(texts, embeddings)

In [10]:
df["topic"] = topics
df.head()

,clean_text,score,bank,topic
0,dikit verivikasi ribet mohon di perbaiki,1,BCAMobile_reviews,0
1,sangat terbantu sexali dg adanya m bangking in,1,BCAMobile_reviews,0
2,kenapa sy tidak bisa mendownlod aplikasi bca,1,BCAMobile_reviews,0
3,sangat kecewa saya melakukan transfer keterang...,1,BCAMobile_reviews,0
4,update an baru ada bugs,1,BCAMobile_reviews,0


In [11]:
topic_model.get_topic_info()

,Topic,Count,Name,Representation,Representative_Docs
0,-1,43,-1_transaksi_brimo_tidak_bisa,"[transaksi, brimo, tidak, bisa, sangat, kecewa...",[ini kenapa tidak bisa buka pake sidik jari ya...
1,0,11945,0_di_bisa_saya_tidak,"[di, bisa, saya, tidak, aplikasi, ini, nya, ad...","[ini kenapa jadi tidak bisa di buka lagi sih, ..."
2,1,12,1_biometrik_login_force_close,"[biometrik, login, force, close, jari, sidik, ...",[mengaktifkan login biometrik keluar sendiri d...


# Evaluating

In [12]:
df["topic"].value_counts().head()

topic
 0    11945
-1       43
 1       12
Name: count, dtype: int64

too many "-1" meaning too many noises => clustering failed

# Tuning

In [13]:
from hdbscan import HDBSCAN

hdbscan_model = HDBSCAN(
  min_cluster_size=50,   # default terlalu kecil/besar tergantung kasus
  min_samples=5,         # lebih longgar
  prediction_data=True
)

In [14]:
topic_model = BERTopic(
  hdbscan_model=hdbscan_model
)

topics, probs = topic_model.fit_transform(texts, embeddings)

In [15]:
df["topic"] = topics
df["topic"].value_counts().head(10)

topic
-1    7038
 0     720
 1     708
 2     476
 3     345
 4     278
 5     265
 6     233
 7     190
 8     179
Name: count, dtype: int64

In [16]:
topic_model.get_topic_info()

,Topic,Count,Name,Representation,Representative_Docs
0,-1,7038,-1_bisa_di_ini_mau,"[bisa, di, ini, mau, nya, aplikasi, saya, kena...",[ini kenapa ya kok tidak bisa masuk ke livinny...
1,0,720,0_saldo_saya_tidak_ada,"[saldo, saya, tidak, ada, tapi, sudah, uang, d...",[kemarin saya melakukan transanksi tranfer ke ...
2,1,708,1_update_mulu_sangat_gangguan,"[update, mulu, sangat, gangguan, membantu, mud...","[tiap bulan update mulu, update mulu ribet, up..."
3,2,476,2_tolong_perbaiki_mohon_di,"[tolong, perbaiki, mohon, di, aplikasi, bisa, ...",[setelah update malah tidak dapat di buka tolo...
4,3,345,3_sy_yg_gk_tp,"[sy, yg, gk, tp, klo, tdk, bs, lg, di, udh]",[knp hrs ada wondr klo bni mobile banking msh ...
5,4,278,4_tunai_tarik_fitur_mobile,"[tunai, tarik, fitur, mobile, bni, tanpa, seto...","[belum ada fitur tarik tunai, fitur mobile tun..."
6,5,265,5_tidak_bisa_buka_di,"[tidak, bisa, buka, di, aplikasi, dibuka, upda...","[sering tidak bisa di buka, aplikasi tidak bis..."
7,6,233,6_jaringan_padahal_sinyal_bagus,"[jaringan, padahal, sinyal, bagus, wifi, inter...",[selalu gangguan jaringan padahal jaringan bag...
8,7,190,7_kode_benar_otp_padahal,"[kode, benar, otp, padahal, login, password, m...",[aku tidak bisa buka aplikasinya kenapa yah pa...
9,8,179,8_wajah_verifikasi_gagal_susah,"[wajah, verifikasi, gagal, susah, ribet, veriv...","[gagal verifikasi wajah mulu, verifikasi wajah..."


still shit , adding UMAP

# UMAP Combination

In [26]:
from umap import UMAP

umap_model = UMAP(
  n_neighbors=5,
  n_components=5,
  min_dist=0.0,
  metric='cosine'
)

In [27]:
topic_model = BERTopic(
  umap_model=umap_model,
  hdbscan_model=hdbscan_model
)

topics, probs = topic_model.fit_transform(texts, embeddings)

In [28]:
df["topic"] = topics
df["topic"].value_counts().head(10)

topic
-1    5703
 0     880
 1     232
 2     218
 3     203
 4     186
 5     172
 6     168
 7     168
 8     166
Name: count, dtype: int64

In [29]:
topic_model.get_topic_info()

,Topic,Count,Name,Representation,Representative_Docs
0,-1,5703,-1_bisa_di_mau_ini,"[bisa, di, mau, ini, aplikasi, nya, saya, gak,...",[awal pake apk ini lancar dan stabil tapi di b...
1,0,880,0_saldo_saya_ada_tapi,"[saldo, saya, ada, tapi, tidak, sudah, uang, d...",[punya pengalaman yg sangat buruk dengan bank ...
2,1,232,1_potongan_rb_bank_biaya,"[potongan, rb, bank, biaya, aja, bca, ribu, sa...",[apaan nih brimo selalu ada potongan hampir ti...
3,2,218,2_update_lemot_sering_malah,"[update, lemot, sering, malah, terus, banget, ...","[sering update tp malah tambah lemot, update t..."
4,3,203,3_bbm_buruk_av_mm,"[bbm, buruk, av, mm, www, jelek, see, nb, oo, ...",[tong iy purwantiningsih o n an do ra san ni n...
5,4,186,4_tunai_tarik_setor_tanpa,"[tunai, tarik, setor, tanpa, kartu, mobile, fi...",[fitur mobile tunai belum tersedia kayanya sep...
6,5,172,5_wajah_verifikasi_gagal_susah,"[wajah, verifikasi, gagal, susah, ampun, aja, ...","[tidak bisa verifikasi wajah gagal mulu, gagal..."
7,6,168,6_tidak_bisa_aplikasi_setelah,"[tidak, bisa, aplikasi, setelah, brimo, dibuka...",[tiba aplikasi tidak bisa dibuka setelah di in...
8,7,168,7_sy_tdk_bs_tp,"[sy, tdk, bs, tp, yg, bni, jg, hrs, klo, jd]",[lampu indikator merah selm hari pdhl internet...
9,8,166,8_udah_verifikasi_wajah_ribet,"[udah, verifikasi, wajah, ribet, banget, gak, ...",[mau daftar susah banget verifikasi muka gagal...


# Analyzing

In [46]:
bca_topics = (
    df[df["bank"] == "BCAMobile_reviews"]["topic"]
    .value_counts()
    .head(3)
)

bca_topics

topic
-1     1538
 0      205
 37      78
Name: count, dtype: int64

In [47]:
for topic_id in bca_topics.index:
    print(f"\nTopic {topic_id}")
    print(topic_model.get_topic(topic_id))


Topic -1
[('bisa', np.float64(0.014708756222363608)), ('di', np.float64(0.01417384324086571)), ('mau', np.float64(0.01185509524173038)), ('ini', np.float64(0.011825445632890175)), ('aplikasi', np.float64(0.011764648248019094)), ('nya', np.float64(0.011683583081447199)), ('saya', np.float64(0.011485825381325327)), ('gak', np.float64(0.010939428861994425)), ('lagi', np.float64(0.010660362085852151)), ('ga', np.float64(0.010477625281031952))]

Topic 0
[('saldo', np.float64(0.02549405804455543)), ('saya', np.float64(0.024746587977365876)), ('ada', np.float64(0.017134098483648916)), ('tapi', np.float64(0.016815760858546136)), ('tidak', np.float64(0.016691779731490938)), ('sudah', np.float64(0.01634798607908862)), ('uang', np.float64(0.015840840037111083)), ('dan', np.float64(0.015653425419931772)), ('di', np.float64(0.015095583060225475)), ('transaksi', np.float64(0.015049810484863898))]

Topic 37
[('merah', np.float64(0.22317503060574784)), ('indikator', np.float64(0.17177727403645662)), 

In [48]:
for topic_id in bca_topics.index:
    print(f"\n=== Topic {topic_id} ===")
    
    print("Keywords:")
    print(topic_model.get_topic(topic_id))
    
    print("\nSample text:")
    print(
        df[(df["bank"] == "BCAMobile_reviews") & (df["topic"] == topic_id)]
        ["clean_text"]
        .head(5)
        .tolist()
    )


=== Topic -1 ===
Keywords:
[('bisa', np.float64(0.014708756222363608)), ('di', np.float64(0.01417384324086571)), ('mau', np.float64(0.01185509524173038)), ('ini', np.float64(0.011825445632890175)), ('aplikasi', np.float64(0.011764648248019094)), ('nya', np.float64(0.011683583081447199)), ('saya', np.float64(0.011485825381325327)), ('gak', np.float64(0.010939428861994425)), ('lagi', np.float64(0.010660362085852151)), ('ga', np.float64(0.010477625281031952))]

Sample text:
['dikit verivikasi ribet mohon di perbaiki', 'sangat kecewa saya melakukan transfer keterangan gagal tapi saldo saya berkurang', 'update an baru ada bugs', 'gimana sih bca pas lagi transfer tiba tiba error dan disuruh cek mutasi padahal yang dibutuhin bukti transfer bukan mutasi di inbox juga bukti transfernya gak ada', 'ga bisa masuk masuk sampe pulsa habiss emangg tolol']

=== Topic 0 ===
Keywords:
[('saldo', np.float64(0.02549405804455543)), ('saya', np.float64(0.024746587977365876)), ('ada', np.float64(0.017134098